# Coordinate geometry

Verify planar frames before interpreting vehicle motion.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## World convention

Heading zero faces +x. Positive yaw turns toward +y.

In [ ]:
from math import pi

import matplotlib.pyplot as plt
import numpy as np

from autonomous_vehicles.simulation.geometry import Pose2, Vec2, interpolate_pose, project_segment
from autonomous_vehicles.simulation.shapes import Rectangle

pose = Pose2(4, 3, pi / 4)
local_points = [Vec2(0, 0), Vec2(3, 0), Vec2(0, 2)]
world_points = [pose.local_to_world(point) for point in local_points]

## Transform points

Local axes rotate and translate together. Inversion recovers the original points.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for local, world in zip(local_points, world_points, strict=True):
    ax.scatter(world.x, world.y)
    ax.text(world.x, world.y, f"local=({local.x}, {local.y})")
    recovered = pose.world_to_local(world)
    np.testing.assert_allclose(recovered.array(), local.array(), atol=1e-12)
ax.set_xlim(0, 9)
ax.set_ylim(0, 9)
ax.set_aspect("equal")
ax.grid()
plt.show()

## Compose poses

Composition applies the child pose in the parent frame.

In [ ]:
child = Pose2(3, 0, -pi / 6)
combined = pose.compose(child)
identity = pose.compose(pose.inverse())
print("Composed:", combined)
print("Inverse roundtrip:", identity)
np.testing.assert_allclose([identity.x, identity.y, identity.yaw], 0, atol=1e-12)

## Project onto a segment

Signed lateral distance is positive to the left of the directed segment.

In [ ]:
start, end = Vec2(0, 0), Vec2(10, 2)
queries = [Vec2(3, 4), Vec2(7, -2), Vec2(-2, 0)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([start.x, end.x], [start.y, end.y], "k-")
for point in queries:
    projection = project_segment(point, start, end)
    ax.plot([point.x, projection.point.x], [point.y, projection.point.y], "o--")
    print(point, projection)
ax.set_aspect("equal")
plt.show()

## Oriented collision geometry

Touching rectangles count as collision. This is geometry, independent of rendering.

In [ ]:
first = Rectangle(Pose2(3, 2, 0.3), 4, 2)
second = Rectangle(Pose2(6, 3, -0.4), 4, 2)
fig, ax = plt.subplots(figsize=(7, 4))
for box, color in ((first, "blue"), (second, "red")):
    corners = box.corners + (box.corners[0],)
    ax.plot([p.x for p in corners], [p.y for p in corners], color=color)
ax.set_aspect("equal")
plt.show()
print("Overlap:", first.overlaps(second))
midpoint = interpolate_pose(Pose2(yaw=pi - 0.1), Pose2(yaw=-pi + 0.1), 0.5)
print("Shortest-angle midpoint:", midpoint.yaw)

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.